# TitanTPP 데이터 특징과 구조 진단

현재 동결 계약의 **train 행만** 분석한다. 성능 비교는 2026-09-28 10:55 KST의 저장된 **validation-only** 집계다. 원격 접속, 모델 학습, checkpoint 추론, held-out row 읽기는 없다. 구조가 성능 차이의 원인이라는 주장은 아직 가설이다.

`README.md`는 해석과 한계, `analyze.py`는 전체 계산, `analysis.json`은 원본 수치다. 아래 실행은 같은 계약/파일의 SHA와 target identity를 재검증한다.

In [1]:
from pathlib import Path
import importlib.util, json
import pandas as pd
from IPython.display import display
ROOT = Path('/Users/igwanhyeong/PycharmProjects/paper_research')
OUT = ROOT / 'reports/titantpp_data_characteristics_20260928_v1'
spec = importlib.util.spec_from_file_location('train_profile', OUT / 'analyze.py')
analysis_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis_module)
result = analysis_module.main()
assert all(d['quality_checks_passed'] and d['target_identity_matches_frozen_contract'] for d in result['datasets'].values())
print('All three frozen train identities verified; no held-out evaluation.')

Profile verified: yellow_trip_hourly
Profile verified: intermittent_frozen_5000
Profile verified: insta_market_basket
Saved /Users/igwanhyeong/PycharmProjects/paper_research/reports/titantpp_data_characteristics_20260928_v1/analysis.json
All three frozen train identities verified; no held-out evaluation.


## 모델이 실제로 보는 이력을 비교한다

수량은 서로 다른 의미·단위다. 이력 개수는 시간 window와 target 포함 최대 길이 제한을 적용한 실제 관측 수다. 상관은 log1p 수량에서 계산한다. 사용자/지역 등의 평균을 제거한 상관은 인과·일반화 성능 검정이 아니다.

In [2]:
rows = []
for name, d in result['datasets'].items():
    rows.append(dict(dataset=name, train_series=d['train_series'], train_targets=d['train_targets'],
                     quantity_median=d['quantity_all_train_rows']['p50'], quantity_p95=d['quantity_all_train_rows']['p95'],
                     history_median=d['actual_target_history_events']['p50'], history_p95=d['actual_target_history_events']['p95'],
                     equal_adjacent_pct=100*d['adjacent_equal_quantity_share'],
                     within_entity_log_lag1_corr=d['adjacent_log_quantity_correlation']['within_entity_demeaned'],
                     history_le7_pct=100*d['target_history_le_shares']['7'], mean_active_branches=d['mean_available_branches']))
profile_table = pd.DataFrame(rows)
display(profile_table)
profile_table.to_csv(OUT / 'dataset_profiles.csv', index=False)

,dataset,train_series,train_targets,quantity_median,quantity_p95,history_median,history_p95,equal_adjacent_pct,within_entity_log_lag1_corr,history_le7_pct,mean_active_branches
0,yellow_trip_hourly,131,38393,7.0,1562.0,129.0,168.0,20.329227,0.749114,2.411898,6.983434
1,intermittent_frozen_5000,5000,393824,2.0,46.0,40.0,144.0,61.989620,0.928121,8.887219,5.565263
2,insta_market_basket,206209,1991192,8.0,25.0,5.0,14.0,10.235226,0.037175,72.134631,2.450429


In [3]:
sensitivity = []
for name, d in result['datasets'].items():
    for cutoff, c in d['adjacent_log_quantity_correlation_long_entities'].items():
        sensitivity.append(dict(dataset=name, minimum_train_events=int(cutoff), pairs=c['pairs'],
                                within_entity_log_correlation=c['within_entity_demeaned']))
display(pd.DataFrame(sensitivity))
# This reduces short-series sensitivity; it does not prove absent nonlinear signal.

,dataset,minimum_train_events,pairs,within_entity_log_correlation
0,yellow_trip_hourly,20,38393,0.749114
1,yellow_trip_hourly,40,38393,0.749114
2,intermittent_frozen_5000,20,393327,0.928121
3,intermittent_frozen_5000,40,390106,0.928168
4,insta_market_basket,20,987123,0.073981
5,insta_market_basket,40,350442,0.050095


## 저장된 validation 비교: 동일 완료 seed만

Taxi는 3 seed가 모두 완료됐다. Intermittent/Instacart의 Full vs MLP 일대일 비교는 현재 seed42만 가능하다. 미완료 조건을 제외해서 최종 결론을 만드는 것이 아니라 미완료 목록을 함께 보존한다. 같은 raw-RMSE-selected checkpoint의 time NLL을 읽는다.

In [4]:
conditions = result['validation']['conditions']
flat = []
strata = []
for r in conditions:
    row = {k:r[k] for k in ['dataset','arm','seed','best_epoch','completed_epochs','parameters','reused']}
    row.update({k:r['selected'][k] for k in ['count','qty_rmse','qty_mae','time_nll']})
    flat.append(row)
    for kind in ['quantity','additional_history']:
        bounds=r['selected'][kind+'_boundaries']
        for cell in r['selected'][kind+'_cells']:
            j=cell['bin']
            strata.append(dict(dataset=r['dataset'], arm=r['arm'], seed=r['seed'], kind=kind,
                               lower_exclusive=bounds[j-1] if j else None,
                               upper_inclusive=bounds[j] if j<len(bounds) else None, **cell))
validation_table = pd.DataFrame(flat)
validation_table.to_csv(OUT / 'validation_conditions.csv', index=False)
pd.DataFrame(strata).to_csv(OUT / 'validation_strata.csv', index=False)
pd.DataFrame(result['validation']['uncompleted_conditions']).to_csv(OUT / 'uncompleted_conditions.csv',index=False)
labels=['titantpp','titantpp_local_detail','titantpp_history_mlp']
matched = validation_table[validation_table.arm.isin(labels) & ((validation_table.dataset=='yellow_trip_hourly') | (validation_table.seed==42))]
display(matched.groupby(['dataset','arm']).agg(n_seeds=('seed','count'),rmse_mean=('qty_rmse','mean'),rmse_sample_sd=('qty_rmse','std'),time_nll=('time_nll','mean')).reset_index())
print('Completed:', len(conditions), 'uncompleted:', len(result['validation']['uncompleted_conditions']))

Completed: 35 uncompleted: 19


,dataset,arm,n_seeds,rmse_mean,rmse_sample_sd,time_nll
0,insta_market_basket,titantpp,1,5.863202,NaN,2.802775
1,insta_market_basket,titantpp_history_mlp,1,5.878730,NaN,2.803218
2,insta_market_basket,titantpp_local_detail,1,5.871375,NaN,2.803848
3,intermittent_frozen_5000,titantpp,1,1.755601,NaN,0.499417
4,intermittent_frozen_5000,titantpp_history_mlp,1,1.578487,NaN,0.346835
5,intermittent_frozen_5000,titantpp_local_detail,1,1.792506,NaN,0.308359
6,yellow_trip_hourly,titantpp,3,90.509264,0.520719,0.724678
7,yellow_trip_hourly,titantpp_history_mlp,3,79.711120,2.825983,1.038668
8,yellow_trip_hourly,titantpp_local_detail,3,78.777378,1.030483,1.182236


## 해석의 한계와 다음 진단

- 변동 크기뿐 아니라 수량 수준·관측 이력 길이·관측 간격·변화 지속성을 함께 본다.
- Raw 수량 통계는 Full의 은닉 표현 차이와 다르다. 은닉 수준/변화의 실제 기여는 별도 진단이 필요하다.
- Train naive 최근값/평균 예시는 기술적 확인이며 validation 벤치마크 점수와 직접 비교하지 않는다.
- 새 replay/학습은 이 노트북에서 실행하지 않는다. 현재 실행을 완료한 뒤 진단 계약으로 범위를 정한다.